# 02 — Calibración del Modelo 1

Recalibra β₁, θ, κ contra los datos reales descargados en Fase 2.

**Secciones:**
1. Filtrado Butterworth de las 4 series
2. Oscilador de Lorenz + alineación al ONI
3. β₁: OLS precipitación vs ONI
4. θ, κ: grid search contra SIMMA
5. **Panel interactivo Plotly** (sensibilidad θ vs κ)
6. Exportar parámetros

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from abm_enso.data import oni as oni_mod, era5, simma as simma_mod
from abm_enso.analysis import filtros, lorenz, metricas, calibracion_beta, calibracion_theta_kappa
from abm_enso.utils.paths import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Filtrado Butterworth

In [ ]:
df_oni = oni_mod.load()['oni'].dropna()
df_era5 = era5.load()
df_simma = simma_mod.load(tipo=['Deslizamiento', 'Flujo'])

print(f'ONI:    {len(df_oni)} meses ({df_oni.index.min():%Y}-{df_oni.index.max():%Y})')
print(f'ERA5:   {len(df_era5)} meses')
print(f'SIMMA:  {len(df_simma)} eventos')

In [ ]:
oni_enso = filtros.butterworth_enso(df_oni)
precip_enso = filtros.filtrar_fuente(df_era5['precip_mm_mes'], quitar_clima=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(df_oni.index, df_oni, color='#999', alpha=0.5, label='ONI crudo')
axes[0].plot(oni_enso.index, oni_enso, color='#1A3A6B', lw=1.5, label='ONI filtrado (2-7 años)')
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_title('Filtrado ENSO: ONI', loc='left', fontweight='bold')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

precip_anom = filtros.desestacionalizar(df_era5['precip_mm_mes'])
axes[1].plot(precip_anom.index, precip_anom, color='#999', alpha=0.5, label='Precip anomalía')
axes[1].plot(precip_enso.index, precip_enso, color='#0F766E', lw=1.5, label='Precip filtrada (2-7 años)')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title('Filtrado ENSO: Precipitación ERA5', loc='left', fontweight='bold')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_filtros_butterworth.png', dpi=150)
plt.show()

## 2. Oscilador de Lorenz

In [ ]:
oni_lorenz = lorenz.generar_oni_sintetico(oni_enso, T=2000.0, seed=42)
r = metricas.pearson_r(oni_lorenz, oni_enso)
print(f'r(Lorenz, ONI_filtrado) = {r:.3f}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(oni_enso.index, oni_enso, color='#1A3A6B', lw=1.2, label='ONI real filtrado')
ax.plot(oni_lorenz.index, oni_lorenz, color='#C05050', lw=1.2, alpha=0.7,
        label=f'ONI Lorenz (r={r:.2f})')
ax.axhline(0, color='k', lw=0.5)
ax.set_title('ONI observado vs sintético (Lorenz)', loc='left', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_lorenz_vs_oni.png', dpi=150)
plt.show()

## 3. β₁ por OLS

In [ ]:
res_beta = calibracion_beta.ols_beta1(df_oni, precip_anom)
for k, v in res_beta.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

# Scatter ONI vs anomalía de precip
aligned = pd.concat([df_oni, precip_anom], axis=1, keys=['oni', 'precip_anom']).dropna()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(aligned['oni'], aligned['precip_anom'], alpha=0.4, s=15, color='#1A3A6B')
x_line = np.linspace(aligned['oni'].min(), aligned['oni'].max(), 100)
y_line = res_beta['alpha'] + res_beta['beta_1'] * x_line
ax.plot(x_line, y_line, color='#C05050', lw=2, label=f"β₁ = {res_beta['beta_1']:.1f}")
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('ONI (°C)'); ax.set_ylabel('Anomalía precip (mm/mes)')
ax.set_title(f'OLS: precip_anom ~ ONI  (r² = {res_beta["r_cuadrado"]:.3f})', fontweight='bold')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_ols_beta1.png', dpi=150)
plt.show()

## 4. Grid search θ, κ contra SIMMA

In [ ]:
# Agregar SIMMA a conteo mensual
simma_mensual = (
    df_simma.dropna(subset=['fecha_evento'])
    .set_index('fecha_evento')
    .resample('MS').size()
)
# Alinear con ERA5
simma_mensual = simma_mensual.reindex(df_era5.index, fill_value=0)

# Evento binario = mes con > 5 eventos
eventos_obs = (simma_mensual > 5).values
precip = df_era5['precip_mm_mes'].values

print(f'Meses con evento SIMMA (>5): {eventos_obs.sum()} / {len(eventos_obs)}')

resultado = calibracion_theta_kappa.grid_search_f1(precip, eventos_obs)
print(f'\nθ* = {resultado.theta_opt:.3f}')
print(f'κ* = {resultado.kappa_opt:.3f}')
print(f'F1 = {resultado.f1_opt:.3f}')

## 5. 🎛️ Panel interactivo Plotly — sensibilidad θ vs κ

Heatmap interactivo: hover para ver F1, click-drag para zoom.

In [ ]:
fig = go.Figure(data=go.Heatmap(
    z=resultado.f1_matrix,
    x=resultado.kappa_grid,
    y=resultado.theta_grid,
    colorscale='Viridis',
    colorbar=dict(title='F1 score'),
    hovertemplate='θ = %{y:.3f}<br>κ = %{x:.3f}<br>F1 = %{z:.3f}<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=[resultado.kappa_opt], y=[resultado.theta_opt],
    mode='markers', marker=dict(size=18, color='red', symbol='x'),
    name=f'Óptimo (F1={resultado.f1_opt:.3f})'
))
fig.update_layout(
    title='Grid search θ, κ (F1 vs SIMMA)',
    xaxis_title='κ (tasa de drenaje mensual)',
    yaxis_title='θ (umbral de activación)',
    width=800, height=600,
)
fig.show()

In [ ]:
# Serie temporal comparando eventos simulados vs observados
eventos_sim = calibracion_theta_kappa.simular_eventos(
    precip, theta=resultado.theta_opt, kappa=resultado.kappa_opt
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_era5.index, y=simma_mensual.values,
    mode='lines', name='Eventos SIMMA/mes',
    line=dict(color='#C05050', width=1.5),
))
fig.add_trace(go.Scatter(
    x=df_era5.index[eventos_sim], y=[simma_mensual.values.max()]*eventos_sim.sum(),
    mode='markers', name=f'Evento simulado (θ={resultado.theta_opt:.2f})',
    marker=dict(color='#1A3A6B', size=6, symbol='triangle-up'),
))
fig.update_layout(
    title=f'Validación: eventos simulados vs observados (F1 = {resultado.f1_opt:.3f})',
    xaxis_title='Fecha', yaxis_title='Eventos SIMMA / mes',
    width=1000, height=400,
)
fig.show()

## 6. Exportar parámetros calibrados

In [ ]:
from abm_enso.pipeline import calibrar_modelo
df_params = calibrar_modelo()
df_params